# MONP Benchmark Audit — Full Pipeline

**Run this notebook in VS Code (Jupyter) to regenerate every figure and table from scratch.**

This single notebook runs the complete analysis end to end — data audit through all main-text
and supplementary outputs — saving every figure as a 300-DPI PNG to `results/figures/` and every
table as a CSV to `results/`, and displaying each one inline as it is produced.

**Before running:** download `dataset.txt` from the NanoTox repository and place it at
`../data/dataset.txt` (see `../data/README.md`). Then choose *Run All*.


## 0. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

import common, figstyle
import duplicate_audit, leakage_curve, lomo_validation, baseline_models
import bootstrap_ci, permutation_test, seed_stability
import feature_decomposition, per_oxide_folds

RESULTS = common.RESULTS_DIR
FIGURES = common.FIGURES_DIR
print('Dataset expected at:', common.DATA_PATH)
print('Results  ->', RESULTS)
print('Figures  ->', FIGURES)


## 1. Data audit  →  Table 2 (dataset composition)
Removes exact duplicates (483 → 364, 68 cytotoxic) and reports per-oxide composition.


In [ ]:
duplicate_audit.main()
display(pd.read_csv(RESULTS / 'per_oxide_composition.csv'))


## 2. Leakage curve  →  Figure 1  +  Table S1
Random k-fold vs LOMO MCC across nine duplication levels.


In [ ]:
leakage_curve.main()
display(pd.read_csv(RESULTS / 'leakage_curve.csv'))
display(Image(str(FIGURES / 'Figure1_leakage_curve.png')))


## 3. LOMO validation  →  Table 1 (five classifiers)


In [ ]:
lomo_validation.main()
display(pd.read_csv(RESULTS / 'classifier_results.csv'))


## 4. Dose-threshold baseline (LOMO)


In [ ]:
baseline_models.main()
display(pd.read_csv(RESULTS / 'baseline_results.csv'))


## 5. Bootstrap confidence intervals  →  Figure 3
Forest plot of LOMO MCC with 95% bootstrap CIs (1,000 resamples).


In [ ]:
bootstrap_ci.main()
display(pd.read_csv(RESULTS / 'bootstrap_results.csv'))
display(Image(str(FIGURES / 'bootstrap_forest.png')))


## 6. Permutation test  →  Figure S1
Null distribution of the XGBoost LOMO MCC under 1,000 label permutations.


In [ ]:
permutation_test.main()
display(pd.read_csv(RESULTS / 'permutation_results.csv'))
display(Image(str(FIGURES / 'FigureS1_permutation_null.png')))


## 7. Seed stability  →  Figure 2  +  Table S2
Per-seed LOMO MCC for each model across eight random seeds.


In [ ]:
seed_stability.main()
display(pd.read_csv(RESULTS / 'seed_stability.csv'))
display(Image(str(FIGURES / 'Figure2_seed_stability.png')))


## 8. Feature-set decomposition  →  Table S4
Full / dose-only / no-dose descriptor sets under LOMO, with CIs.


In [ ]:
feature_decomposition.main()
display(pd.read_csv(RESULTS / 'feature_decomposition.csv'))


## 9. Per-oxide LOMO folds  →  Table S3  +  Figure S2
Per-oxide held-out performance with n and cytotoxic counts.


In [ ]:
per_oxide_folds.main()
display(pd.read_csv(RESULTS / 'per_oxide_folds.csv'))
display(Image(str(FIGURES / 'FigureS2_per_oxide_folds.png')))


## 11. Self-check  (computed vs manuscript)
Confirms the run reproduces the reported dataset composition.


In [ ]:
raw = common.load_raw(); dedup = common.deduplicate(raw); y = common.binary_labels(dedup)
checks = [('raw_records', len(raw), 483), ('unique_records', len(dedup), 364), ('cytotoxic', int(y.sum()), 68)]
ok = True
for name, got, exp in checks:
    status = 'OK' if got == exp else 'MISMATCH'
    ok = ok and status == 'OK'
    print(f'[{status:8s}] {name:16s} computed={got}  expected={exp}')
print()
print('Self-check PASSED.' if ok else 'Self-check MISMATCH — check data path / column names.')
print('Figures (300 DPI):', FIGURES)
print('Tables (CSV):     ', RESULTS)
